In [0]:
%sql
-- Databricks SQL
-- Wide feature table for monthly SUSEP claim forecasting
-- Granularity: year/month
-- Objective: predict vl_sinistro for next month, without company or insurance line dimensions

CREATE SCHEMA IF NOT EXISTS susep_gold;

CREATE OR REPLACE TABLE susep_gold.wt_ml_sinistro_mensal
USING DELTA
COMMENT 'Wide feature table for monthly claim forecasting using SUSEP insurance data aggregated by year/month'
AS

WITH base_raw AS (

    -- Original monthly aggregation
    -- This is the source query requested by the project
    SELECT 
        nr_ano_mes_referencia,
        SUM(vl_sinistro_ocorrido) AS vl_sinistro,
        SUM(vl_premio_ganho)      AS vl_premio
    FROM susep_silver.vw_fato_seguro_mensal_susep
    GROUP BY 
        nr_ano_mes_referencia
    ORDER BY 
        nr_ano_mes_referencia DESC
    LIMIT 121

),

base_monthly AS (

    SELECT
        CAST(nr_ano_mes_referencia AS INT) AS nr_ano_mes_referencia,

        TO_DATE(
            CONCAT(CAST(nr_ano_mes_referencia AS STRING), '01'),
            'yyyyMMdd'
        ) AS dt_mes_referencia,

        CAST(vl_sinistro AS DOUBLE) AS vl_sinistro,
        CAST(vl_premio   AS DOUBLE) AS vl_premio

    FROM base_raw

),

base_ordered AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            ORDER BY dt_mes_referencia
        ) AS nr_indice_tempo,

        COUNT(*) OVER () AS qt_total_meses,

        MIN(dt_mes_referencia) OVER () AS dt_primeiro_mes_base,
        MAX(dt_mes_referencia) OVER () AS dt_ultimo_mes_base

    FROM base_monthly

),

base_enriched AS (

    SELECT
        nr_ano_mes_referencia,
        dt_mes_referencia,

        CAST(DATE_FORMAT(ADD_MONTHS(dt_mes_referencia, 1), 'yyyyMM') AS INT) AS nr_ano_mes_previsao,
        ADD_MONTHS(dt_mes_referencia, 1) AS dt_mes_previsao,

        YEAR(dt_mes_referencia) AS nr_ano_referencia,
        MONTH(dt_mes_referencia) AS nr_mes_referencia,
        QUARTER(dt_mes_referencia) AS nr_trimestre_referencia,

        YEAR(ADD_MONTHS(dt_mes_referencia, 1)) AS nr_ano_previsao,
        MONTH(ADD_MONTHS(dt_mes_referencia, 1)) AS nr_mes_previsao,
        QUARTER(ADD_MONTHS(dt_mes_referencia, 1)) AS nr_trimestre_previsao,

        CASE 
            WHEN MONTH(dt_mes_referencia) BETWEEN 1 AND 6 THEN 1 
            ELSE 2 
        END AS nr_semestre_referencia,

        CASE 
            WHEN MONTH(ADD_MONTHS(dt_mes_referencia, 1)) BETWEEN 1 AND 6 THEN 1 
            ELSE 2 
        END AS nr_semestre_previsao,

        nr_indice_tempo,
        nr_indice_tempo + 1 AS nr_indice_tempo_previsao,
        POWER(nr_indice_tempo + 1, 2) AS nr_indice_tempo_previsao_sq,

        qt_total_meses,
        dt_primeiro_mes_base,
        dt_ultimo_mes_base,

        vl_sinistro,
        vl_premio,

        CASE 
            WHEN vl_premio IS NULL OR vl_premio = 0 THEN NULL
            ELSE vl_sinistro / vl_premio
        END AS vl_sinistralidade,

        LOG1P(vl_sinistro) AS ln_vl_sinistro,
        LOG1P(vl_premio)   AS ln_vl_premio

    FROM base_ordered

),

target_features AS (

    SELECT
        *,

        -- Target: next month claim amount
        LEAD(vl_sinistro, 1) OVER (
            ORDER BY dt_mes_referencia
        ) AS vl_sinistro_target_prox_mes,

        -- Log target, useful for Linear Regression, Ridge, Lasso, XGBoost and LightGBM
        LOG1P(
            LEAD(vl_sinistro, 1) OVER (
                ORDER BY dt_mes_referencia
            )
        ) AS ln_vl_sinistro_target_prox_mes

    FROM base_enriched

),

lag_features AS (

    SELECT
        *,

        -- Current month values.
        -- For a t+1 forecast, these are valid features because month t is already known.
        vl_sinistro AS vl_sinistro_lag_0,
        vl_premio AS vl_premio_lag_0,
        vl_sinistralidade AS vl_sinistralidade_lag_0,

        -- Claim lags
        LAG(vl_sinistro, 1)  OVER (ORDER BY dt_mes_referencia) AS vl_sinistro_lag_1,
        LAG(vl_sinistro, 2)  OVER (ORDER BY dt_mes_referencia) AS vl_sinistro_lag_2,
        LAG(vl_sinistro, 3)  OVER (ORDER BY dt_mes_referencia) AS vl_sinistro_lag_3,
        LAG(vl_sinistro, 4)  OVER (ORDER BY dt_mes_referencia) AS vl_sinistro_lag_4,
        LAG(vl_sinistro, 5)  OVER (ORDER BY dt_mes_referencia) AS vl_sinistro_lag_5,
        LAG(vl_sinistro, 6)  OVER (ORDER BY dt_mes_referencia) AS vl_sinistro_lag_6,
        LAG(vl_sinistro, 11) OVER (ORDER BY dt_mes_referencia) AS vl_sinistro_lag_11,
        LAG(vl_sinistro, 12) OVER (ORDER BY dt_mes_referencia) AS vl_sinistro_lag_12,
        LAG(vl_sinistro, 13) OVER (ORDER BY dt_mes_referencia) AS vl_sinistro_lag_13,
        LAG(vl_sinistro, 24) OVER (ORDER BY dt_mes_referencia) AS vl_sinistro_lag_24,

        -- Premium lags
        LAG(vl_premio, 1)  OVER (ORDER BY dt_mes_referencia) AS vl_premio_lag_1,
        LAG(vl_premio, 2)  OVER (ORDER BY dt_mes_referencia) AS vl_premio_lag_2,
        LAG(vl_premio, 3)  OVER (ORDER BY dt_mes_referencia) AS vl_premio_lag_3,
        LAG(vl_premio, 4)  OVER (ORDER BY dt_mes_referencia) AS vl_premio_lag_4,
        LAG(vl_premio, 5)  OVER (ORDER BY dt_mes_referencia) AS vl_premio_lag_5,
        LAG(vl_premio, 6)  OVER (ORDER BY dt_mes_referencia) AS vl_premio_lag_6,
        LAG(vl_premio, 11) OVER (ORDER BY dt_mes_referencia) AS vl_premio_lag_11,
        LAG(vl_premio, 12) OVER (ORDER BY dt_mes_referencia) AS vl_premio_lag_12,
        LAG(vl_premio, 13) OVER (ORDER BY dt_mes_referencia) AS vl_premio_lag_13,
        LAG(vl_premio, 24) OVER (ORDER BY dt_mes_referencia) AS vl_premio_lag_24,

        -- Loss ratio lags
        LAG(vl_sinistralidade, 1)  OVER (ORDER BY dt_mes_referencia) AS vl_sinistralidade_lag_1,
        LAG(vl_sinistralidade, 2)  OVER (ORDER BY dt_mes_referencia) AS vl_sinistralidade_lag_2,
        LAG(vl_sinistralidade, 3)  OVER (ORDER BY dt_mes_referencia) AS vl_sinistralidade_lag_3,
        LAG(vl_sinistralidade, 6)  OVER (ORDER BY dt_mes_referencia) AS vl_sinistralidade_lag_6,
        LAG(vl_sinistralidade, 11) OVER (ORDER BY dt_mes_referencia) AS vl_sinistralidade_lag_11,
        LAG(vl_sinistralidade, 12) OVER (ORDER BY dt_mes_referencia) AS vl_sinistralidade_lag_12,
        LAG(vl_sinistralidade, 24) OVER (ORDER BY dt_mes_referencia) AS vl_sinistralidade_lag_24,

        -- Log lags, useful for linear models and models with multiplicative behavior
        LAG(ln_vl_sinistro, 1)  OVER (ORDER BY dt_mes_referencia) AS ln_vl_sinistro_lag_1,
        LAG(ln_vl_sinistro, 3)  OVER (ORDER BY dt_mes_referencia) AS ln_vl_sinistro_lag_3,
        LAG(ln_vl_sinistro, 6)  OVER (ORDER BY dt_mes_referencia) AS ln_vl_sinistro_lag_6,
        LAG(ln_vl_sinistro, 11) OVER (ORDER BY dt_mes_referencia) AS ln_vl_sinistro_lag_11,
        LAG(ln_vl_sinistro, 12) OVER (ORDER BY dt_mes_referencia) AS ln_vl_sinistro_lag_12,

        LAG(ln_vl_premio, 1)  OVER (ORDER BY dt_mes_referencia) AS ln_vl_premio_lag_1,
        LAG(ln_vl_premio, 3)  OVER (ORDER BY dt_mes_referencia) AS ln_vl_premio_lag_3,
        LAG(ln_vl_premio, 6)  OVER (ORDER BY dt_mes_referencia) AS ln_vl_premio_lag_6,
        LAG(ln_vl_premio, 11) OVER (ORDER BY dt_mes_referencia) AS ln_vl_premio_lag_11,
        LAG(ln_vl_premio, 12) OVER (ORDER BY dt_mes_referencia) AS ln_vl_premio_lag_12

    FROM target_features

),

rolling_features AS (

    SELECT
        *,

        -- Rolling claim features using information available until reference month t
        AVG(vl_sinistro) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS vl_sinistro_media_3m,

        AVG(vl_sinistro) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS vl_sinistro_media_6m,

        AVG(vl_sinistro) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS vl_sinistro_media_12m,

        AVG(vl_sinistro) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 23 PRECEDING AND CURRENT ROW
        ) AS vl_sinistro_media_24m,

        STDDEV_SAMP(vl_sinistro) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS vl_sinistro_std_3m,

        STDDEV_SAMP(vl_sinistro) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS vl_sinistro_std_6m,

        STDDEV_SAMP(vl_sinistro) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS vl_sinistro_std_12m,

        MIN(vl_sinistro) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS vl_sinistro_min_12m,

        MAX(vl_sinistro) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS vl_sinistro_max_12m,

        -- Previous 12-month window excluding the current month.
        -- Useful to detect whether the current month is unusual compared to recent history.
        AVG(vl_sinistro) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 12 PRECEDING AND 1 PRECEDING
        ) AS vl_sinistro_media_12m_excl_mes_atual,

        STDDEV_SAMP(vl_sinistro) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 12 PRECEDING AND 1 PRECEDING
        ) AS vl_sinistro_std_12m_excl_mes_atual,

        -- Rolling premium features
        AVG(vl_premio) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS vl_premio_media_3m,

        AVG(vl_premio) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS vl_premio_media_6m,

        AVG(vl_premio) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS vl_premio_media_12m,

        AVG(vl_premio) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 23 PRECEDING AND CURRENT ROW
        ) AS vl_premio_media_24m,

        STDDEV_SAMP(vl_premio) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS vl_premio_std_12m,

        MIN(vl_premio) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS vl_premio_min_12m,

        MAX(vl_premio) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS vl_premio_max_12m,

        -- Rolling loss ratio features
        AVG(vl_sinistralidade) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS vl_sinistralidade_media_3m,

        AVG(vl_sinistralidade) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS vl_sinistralidade_media_6m,

        AVG(vl_sinistralidade) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS vl_sinistralidade_media_12m,

        STDDEV_SAMP(vl_sinistralidade) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS vl_sinistralidade_std_12m,

        -- Rolling log features
        AVG(ln_vl_sinistro) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS ln_vl_sinistro_media_3m,

        AVG(ln_vl_sinistro) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS ln_vl_sinistro_media_6m,

        AVG(ln_vl_sinistro) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS ln_vl_sinistro_media_12m,

        AVG(ln_vl_premio) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS ln_vl_premio_media_3m,

        AVG(ln_vl_premio) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS ln_vl_premio_media_6m,

        AVG(ln_vl_premio) OVER (
            ORDER BY dt_mes_referencia
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS ln_vl_premio_media_12m

    FROM lag_features

),

variation_features AS (

    SELECT
        *,

        -- Claim variations
        CASE 
            WHEN vl_sinistro_lag_1 IS NULL OR vl_sinistro_lag_1 = 0 THEN NULL
            ELSE (vl_sinistro / vl_sinistro_lag_1) - 1
        END AS pc_sinistro_mom,

        CASE 
            WHEN vl_sinistro_lag_3 IS NULL OR vl_sinistro_lag_3 = 0 THEN NULL
            ELSE (vl_sinistro / vl_sinistro_lag_3) - 1
        END AS pc_sinistro_var_3m,

        CASE 
            WHEN vl_sinistro_lag_6 IS NULL OR vl_sinistro_lag_6 = 0 THEN NULL
            ELSE (vl_sinistro / vl_sinistro_lag_6) - 1
        END AS pc_sinistro_var_6m,

        CASE 
            WHEN vl_sinistro_lag_12 IS NULL OR vl_sinistro_lag_12 = 0 THEN NULL
            ELSE (vl_sinistro / vl_sinistro_lag_12) - 1
        END AS pc_sinistro_yoy_mes_referencia,

        -- This compares current month t with t-11.
        -- It is useful because t-11 is the same calendar month of the target t+1 in the previous year.
        CASE 
            WHEN vl_sinistro_lag_11 IS NULL OR vl_sinistro_lag_11 = 0 THEN NULL
            ELSE (vl_sinistro / vl_sinistro_lag_11) - 1
        END AS pc_sinistro_vs_mes_previsao_ano_anterior,

        -- Premium variations
        CASE 
            WHEN vl_premio_lag_1 IS NULL OR vl_premio_lag_1 = 0 THEN NULL
            ELSE (vl_premio / vl_premio_lag_1) - 1
        END AS pc_premio_mom,

        CASE 
            WHEN vl_premio_lag_3 IS NULL OR vl_premio_lag_3 = 0 THEN NULL
            ELSE (vl_premio / vl_premio_lag_3) - 1
        END AS pc_premio_var_3m,

        CASE 
            WHEN vl_premio_lag_6 IS NULL OR vl_premio_lag_6 = 0 THEN NULL
            ELSE (vl_premio / vl_premio_lag_6) - 1
        END AS pc_premio_var_6m,

        CASE 
            WHEN vl_premio_lag_12 IS NULL OR vl_premio_lag_12 = 0 THEN NULL
            ELSE (vl_premio / vl_premio_lag_12) - 1
        END AS pc_premio_yoy_mes_referencia,

        -- Loss ratio variations
        CASE 
            WHEN vl_sinistralidade_lag_1 IS NULL OR vl_sinistralidade_lag_1 = 0 THEN NULL
            ELSE (vl_sinistralidade / vl_sinistralidade_lag_1) - 1
        END AS pc_sinistralidade_mom,

        CASE 
            WHEN vl_sinistralidade_lag_12 IS NULL OR vl_sinistralidade_lag_12 = 0 THEN NULL
            ELSE (vl_sinistralidade / vl_sinistralidade_lag_12) - 1
        END AS pc_sinistralidade_yoy_mes_referencia,

        -- Absolute deltas
        vl_sinistro - vl_sinistro_lag_1  AS vl_delta_sinistro_1m,
        vl_sinistro - vl_sinistro_lag_3  AS vl_delta_sinistro_3m,
        vl_sinistro - vl_sinistro_lag_6  AS vl_delta_sinistro_6m,
        vl_sinistro - vl_sinistro_lag_12 AS vl_delta_sinistro_12m,

        vl_premio - vl_premio_lag_1  AS vl_delta_premio_1m,
        vl_premio - vl_premio_lag_3  AS vl_delta_premio_3m,
        vl_premio - vl_premio_lag_6  AS vl_delta_premio_6m,
        vl_premio - vl_premio_lag_12 AS vl_delta_premio_12m,

        -- Distance from rolling average
        vl_sinistro - vl_sinistro_media_3m  AS vl_sinistro_gap_media_3m,
        vl_sinistro - vl_sinistro_media_6m  AS vl_sinistro_gap_media_6m,
        vl_sinistro - vl_sinistro_media_12m AS vl_sinistro_gap_media_12m,

        CASE 
            WHEN vl_sinistro_media_12m IS NULL OR vl_sinistro_media_12m = 0 THEN NULL
            ELSE (vl_sinistro / vl_sinistro_media_12m) - 1
        END AS pc_sinistro_gap_media_12m,

        CASE 
            WHEN vl_premio_media_12m IS NULL OR vl_premio_media_12m = 0 THEN NULL
            ELSE (vl_premio / vl_premio_media_12m) - 1
        END AS pc_premio_gap_media_12m,

        CASE 
            WHEN vl_sinistralidade_media_12m IS NULL OR vl_sinistralidade_media_12m = 0 THEN NULL
            ELSE (vl_sinistralidade / vl_sinistralidade_media_12m) - 1
        END AS pc_sinistralidade_gap_media_12m

    FROM rolling_features

),

calendar_features AS (

    SELECT
        *,

        -- Cyclical representation of the forecast month.
        -- This is the recommended calendar feature because the target is t+1.
        SIN(2 * PI() * nr_mes_previsao / 12) AS ft_mes_previsao_sin,
        COS(2 * PI() * nr_mes_previsao / 12) AS ft_mes_previsao_cos,

        -- Cyclical representation of the reference month.
        -- Useful as support feature, but forecast month is usually more important.
        SIN(2 * PI() * nr_mes_referencia / 12) AS ft_mes_referencia_sin,
        COS(2 * PI() * nr_mes_referencia / 12) AS ft_mes_referencia_cos,

        -- Forecast month one-hot flags
        CASE WHEN nr_mes_previsao = 1  THEN 1 ELSE 0 END AS fl_mes_previsao_janeiro,
        CASE WHEN nr_mes_previsao = 2  THEN 1 ELSE 0 END AS fl_mes_previsao_fevereiro,
        CASE WHEN nr_mes_previsao = 3  THEN 1 ELSE 0 END AS fl_mes_previsao_marco,
        CASE WHEN nr_mes_previsao = 4  THEN 1 ELSE 0 END AS fl_mes_previsao_abril,
        CASE WHEN nr_mes_previsao = 5  THEN 1 ELSE 0 END AS fl_mes_previsao_maio,
        CASE WHEN nr_mes_previsao = 6  THEN 1 ELSE 0 END AS fl_mes_previsao_junho,
        CASE WHEN nr_mes_previsao = 7  THEN 1 ELSE 0 END AS fl_mes_previsao_julho,
        CASE WHEN nr_mes_previsao = 8  THEN 1 ELSE 0 END AS fl_mes_previsao_agosto,
        CASE WHEN nr_mes_previsao = 9  THEN 1 ELSE 0 END AS fl_mes_previsao_setembro,
        CASE WHEN nr_mes_previsao = 10 THEN 1 ELSE 0 END AS fl_mes_previsao_outubro,
        CASE WHEN nr_mes_previsao = 11 THEN 1 ELSE 0 END AS fl_mes_previsao_novembro,
        CASE WHEN nr_mes_previsao = 12 THEN 1 ELSE 0 END AS fl_mes_previsao_dezembro,

        -- Forecast quarter one-hot flags
        CASE WHEN nr_trimestre_previsao = 1 THEN 1 ELSE 0 END AS fl_trimestre_previsao_1,
        CASE WHEN nr_trimestre_previsao = 2 THEN 1 ELSE 0 END AS fl_trimestre_previsao_2,
        CASE WHEN nr_trimestre_previsao = 3 THEN 1 ELSE 0 END AS fl_trimestre_previsao_3,
        CASE WHEN nr_trimestre_previsao = 4 THEN 1 ELSE 0 END AS fl_trimestre_previsao_4

    FROM variation_features

),

outlier_features AS (

    SELECT
        *,

        CASE 
            WHEN vl_sinistro_std_12m_excl_mes_atual IS NULL 
              OR vl_sinistro_std_12m_excl_mes_atual = 0 
            THEN NULL
            ELSE 
                (vl_sinistro - vl_sinistro_media_12m_excl_mes_atual) 
                / vl_sinistro_std_12m_excl_mes_atual
        END AS vl_sinistro_zscore_12m_excl_mes_atual,

        CASE 
            WHEN vl_sinistro_std_12m_excl_mes_atual IS NULL 
              OR vl_sinistro_std_12m_excl_mes_atual = 0 
            THEN 0
            WHEN ABS(
                (vl_sinistro - vl_sinistro_media_12m_excl_mes_atual) 
                / vl_sinistro_std_12m_excl_mes_atual
            ) >= 2.5 THEN 1
            ELSE 0
        END AS fl_sinistro_outlier_12m,

        CASE 
            WHEN pc_sinistro_mom IS NOT NULL AND ABS(pc_sinistro_mom) >= 0.30 THEN 1
            ELSE 0
        END AS fl_variacao_sinistro_mom_alta,

        CASE 
            WHEN pc_sinistro_yoy_mes_referencia IS NOT NULL 
             AND ABS(pc_sinistro_yoy_mes_referencia) >= 0.30 THEN 1
            ELSE 0
        END AS fl_variacao_sinistro_yoy_alta

    FROM calendar_features

),

baseline_features AS (

    SELECT
        *,

        -- Baseline 1: next month equals the last observed month
        vl_sinistro AS vl_pred_baseline_ultimo_mes,

        -- Baseline 2: next month equals the same calendar month from the previous year.
        -- Because target is t+1, the equivalent month from last year is t-11.
        vl_sinistro_lag_11 AS vl_pred_baseline_sazonal_ano_anterior,

        -- Baseline 3: rolling averages
        vl_sinistro_media_3m  AS vl_pred_baseline_media_3m,
        vl_sinistro_media_6m  AS vl_pred_baseline_media_6m,
        vl_sinistro_media_12m AS vl_pred_baseline_media_12m,

        -- Baseline 4: weighted recent average
        CASE
            WHEN vl_sinistro_lag_1 IS NULL OR vl_sinistro_lag_2 IS NULL THEN NULL
            ELSE 
                (0.50 * vl_sinistro) +
                (0.30 * vl_sinistro_lag_1) +
                (0.20 * vl_sinistro_lag_2)
        END AS vl_pred_baseline_media_ponderada_3m,

        -- Log versions of baseline predictions
        LOG1P(vl_sinistro) AS ln_pred_baseline_ultimo_mes,

        CASE 
            WHEN vl_sinistro_lag_11 IS NULL THEN NULL
            ELSE LOG1P(vl_sinistro_lag_11)
        END AS ln_pred_baseline_sazonal_ano_anterior,

        CASE 
            WHEN vl_sinistro_media_3m IS NULL THEN NULL
            ELSE LOG1P(vl_sinistro_media_3m)
        END AS ln_pred_baseline_media_3m,

        CASE 
            WHEN vl_sinistro_media_6m IS NULL THEN NULL
            ELSE LOG1P(vl_sinistro_media_6m)
        END AS ln_pred_baseline_media_6m,

        CASE 
            WHEN vl_sinistro_media_12m IS NULL THEN NULL
            ELSE LOG1P(vl_sinistro_media_12m)
        END AS ln_pred_baseline_media_12m

    FROM outlier_features

),

model_ready_flags AS (

    SELECT
        *,

        CASE 
            WHEN vl_sinistro_target_prox_mes IS NULL THEN 0
            ELSE 1
        END AS fl_linha_treino,

        CASE 
            WHEN vl_sinistro_target_prox_mes IS NULL 
             AND dt_mes_referencia = dt_ultimo_mes_base THEN 1
            ELSE 0
        END AS fl_linha_previsao,

        -- Minimum history for simple models and short lags
        CASE
            WHEN vl_sinistro_target_prox_mes IS NOT NULL
             AND vl_sinistro_lag_3 IS NOT NULL
             AND vl_premio_lag_3 IS NOT NULL
            THEN 1
            ELSE 0
        END AS fl_modelagem_minima,

        -- Recommended flag for MVP model training.
        -- Requires enough history for seasonality and rolling 12-month features.
        CASE
            WHEN vl_sinistro_target_prox_mes IS NOT NULL
             AND vl_sinistro_lag_11 IS NOT NULL
             AND vl_sinistro_lag_12 IS NOT NULL
             AND vl_premio_lag_12 IS NOT NULL
             AND vl_sinistralidade_lag_12 IS NOT NULL
             AND vl_sinistro_media_12m IS NOT NULL
             AND vl_premio_media_12m IS NOT NULL
            THEN 1
            ELSE 0
        END AS fl_modelagem_12m,

        -- More restrictive flag for models using 24-month features
        CASE
            WHEN vl_sinistro_target_prox_mes IS NOT NULL
             AND vl_sinistro_lag_24 IS NOT NULL
             AND vl_premio_lag_24 IS NOT NULL
             AND vl_sinistralidade_lag_24 IS NOT NULL
             AND vl_sinistro_media_24m IS NOT NULL
             AND vl_premio_media_24m IS NOT NULL
            THEN 1
            ELSE 0
        END AS fl_modelagem_24m

    FROM baseline_features

),

final_table AS (

    SELECT
        -- Identifiers
        nr_ano_mes_referencia,
        dt_mes_referencia,
        nr_ano_mes_previsao,
        dt_mes_previsao,

        -- Calendar and trend features
        nr_ano_referencia,
        nr_mes_referencia,
        nr_trimestre_referencia,
        nr_semestre_referencia,

        nr_ano_previsao,
        nr_mes_previsao,
        nr_trimestre_previsao,
        nr_semestre_previsao,

        nr_indice_tempo,
        nr_indice_tempo_previsao,
        nr_indice_tempo_previsao_sq,

        ft_mes_previsao_sin,
        ft_mes_previsao_cos,
        ft_mes_referencia_sin,
        ft_mes_referencia_cos,

        fl_mes_previsao_janeiro,
        fl_mes_previsao_fevereiro,
        fl_mes_previsao_marco,
        fl_mes_previsao_abril,
        fl_mes_previsao_maio,
        fl_mes_previsao_junho,
        fl_mes_previsao_julho,
        fl_mes_previsao_agosto,
        fl_mes_previsao_setembro,
        fl_mes_previsao_outubro,
        fl_mes_previsao_novembro,
        fl_mes_previsao_dezembro,

        fl_trimestre_previsao_1,
        fl_trimestre_previsao_2,
        fl_trimestre_previsao_3,
        fl_trimestre_previsao_4,

        -- Current observed values
        vl_sinistro,
        vl_premio,
        vl_sinistralidade,
        ln_vl_sinistro,
        ln_vl_premio,

        -- Claim lags
        vl_sinistro_lag_0,
        vl_sinistro_lag_1,
        vl_sinistro_lag_2,
        vl_sinistro_lag_3,
        vl_sinistro_lag_4,
        vl_sinistro_lag_5,
        vl_sinistro_lag_6,
        vl_sinistro_lag_11,
        vl_sinistro_lag_12,
        vl_sinistro_lag_13,
        vl_sinistro_lag_24,

        -- Premium lags
        vl_premio_lag_0,
        vl_premio_lag_1,
        vl_premio_lag_2,
        vl_premio_lag_3,
        vl_premio_lag_4,
        vl_premio_lag_5,
        vl_premio_lag_6,
        vl_premio_lag_11,
        vl_premio_lag_12,
        vl_premio_lag_13,
        vl_premio_lag_24,

        -- Loss ratio lags
        vl_sinistralidade_lag_0,
        vl_sinistralidade_lag_1,
        vl_sinistralidade_lag_2,
        vl_sinistralidade_lag_3,
        vl_sinistralidade_lag_6,
        vl_sinistralidade_lag_11,
        vl_sinistralidade_lag_12,
        vl_sinistralidade_lag_24,

        -- Log lags
        ln_vl_sinistro_lag_1,
        ln_vl_sinistro_lag_3,
        ln_vl_sinistro_lag_6,
        ln_vl_sinistro_lag_11,
        ln_vl_sinistro_lag_12,

        ln_vl_premio_lag_1,
        ln_vl_premio_lag_3,
        ln_vl_premio_lag_6,
        ln_vl_premio_lag_11,
        ln_vl_premio_lag_12,

        -- Rolling claim features
        vl_sinistro_media_3m,
        vl_sinistro_media_6m,
        vl_sinistro_media_12m,
        vl_sinistro_media_24m,

        vl_sinistro_std_3m,
        vl_sinistro_std_6m,
        vl_sinistro_std_12m,

        vl_sinistro_min_12m,
        vl_sinistro_max_12m,

        vl_sinistro_media_12m_excl_mes_atual,
        vl_sinistro_std_12m_excl_mes_atual,

        -- Rolling premium features
        vl_premio_media_3m,
        vl_premio_media_6m,
        vl_premio_media_12m,
        vl_premio_media_24m,

        vl_premio_std_12m,
        vl_premio_min_12m,
        vl_premio_max_12m,

        -- Rolling loss ratio features
        vl_sinistralidade_media_3m,
        vl_sinistralidade_media_6m,
        vl_sinistralidade_media_12m,
        vl_sinistralidade_std_12m,

        -- Rolling log features
        ln_vl_sinistro_media_3m,
        ln_vl_sinistro_media_6m,
        ln_vl_sinistro_media_12m,

        ln_vl_premio_media_3m,
        ln_vl_premio_media_6m,
        ln_vl_premio_media_12m,

        -- Variation features
        pc_sinistro_mom,
        pc_sinistro_var_3m,
        pc_sinistro_var_6m,
        pc_sinistro_yoy_mes_referencia,
        pc_sinistro_vs_mes_previsao_ano_anterior,

        pc_premio_mom,
        pc_premio_var_3m,
        pc_premio_var_6m,
        pc_premio_yoy_mes_referencia,

        pc_sinistralidade_mom,
        pc_sinistralidade_yoy_mes_referencia,

        -- Delta features
        vl_delta_sinistro_1m,
        vl_delta_sinistro_3m,
        vl_delta_sinistro_6m,
        vl_delta_sinistro_12m,

        vl_delta_premio_1m,
        vl_delta_premio_3m,
        vl_delta_premio_6m,
        vl_delta_premio_12m,

        -- Gap features
        vl_sinistro_gap_media_3m,
        vl_sinistro_gap_media_6m,
        vl_sinistro_gap_media_12m,

        pc_sinistro_gap_media_12m,
        pc_premio_gap_media_12m,
        pc_sinistralidade_gap_media_12m,

        -- Outlier features
        vl_sinistro_zscore_12m_excl_mes_atual,
        fl_sinistro_outlier_12m,
        fl_variacao_sinistro_mom_alta,
        fl_variacao_sinistro_yoy_alta,

        -- Baseline predictions
        vl_pred_baseline_ultimo_mes,
        vl_pred_baseline_sazonal_ano_anterior,
        vl_pred_baseline_media_3m,
        vl_pred_baseline_media_6m,
        vl_pred_baseline_media_12m,
        vl_pred_baseline_media_ponderada_3m,

        ln_pred_baseline_ultimo_mes,
        ln_pred_baseline_sazonal_ano_anterior,
        ln_pred_baseline_media_3m,
        ln_pred_baseline_media_6m,
        ln_pred_baseline_media_12m,

        -- Target
        vl_sinistro_target_prox_mes,
        ln_vl_sinistro_target_prox_mes,

        -- Control flags
        fl_linha_treino,
        fl_linha_previsao,
        fl_modelagem_minima,
        fl_modelagem_12m,
        fl_modelagem_24m,

        qt_total_meses,
        dt_primeiro_mes_base,
        dt_ultimo_mes_base,

        CURRENT_TIMESTAMP() AS dh_processamento

    FROM model_ready_flags

)

SELECT *
FROM final_table
ORDER BY dt_mes_referencia;